# Домашнее задание 3. Статистическая проверка качества регрессии.

### 1. Загрузите файл candidates.csv с компьютера. 

* откройте первые несколько строк
* изучите строение файла.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
from sklearn.model_selection import train_test_split

data = pd.read_csv('candidates.csv', sep=';', decimal=',', encoding='utf-8-sig')
data.head(10)

,Код претендента,Рейтинг университета окончания,Средний балл диплома,Опыт работы (лет),Возраст,Рейтинг университета диплома,Баллы за сертификаты,Желаемая заработная плата,Итоговая зарплата от компании
0,1001,82,3.22,10,33,84,89,237000,237000
1,1002,4,3.19,3,29,16,0,155000,155000
2,1003,70,3.84,6,30,70,74,212000,197000
3,1004,98,3.67,2,24,97,64,140000,130000
4,1005,94,3.92,2,27,99,66,146000,136000
5,1006,6,4.32,11,33,9,32,322000,299000
6,1007,47,3.33,5,29,43,0,192000,177000
7,1008,32,3.33,5,31,20,61,242000,216000
8,1009,30,4.64,11,36,29,60,307000,307000
9,1010,83,3.92,4,27,88,88,175000,175000


In [2]:
print(data.shape)
print(data.dtypes)
print('пропуски:', data.isnull().values.any())

(800, 9)
Код претендента                     int64
Рейтинг университета окончания      int64
Средний балл диплома              float64
Опыт работы (лет)                   int64
Возраст                             int64
Рейтинг университета диплома        int64
Баллы за сертификаты                int64
Желаемая заработная плата           int64
Итоговая зарплата от компании       int64
dtype: object
пропуски: False


В файле 800 кандидатов и 9 колонок, разделитель `;`, дробные числа записаны через запятую (поэтому `decimal=','`). Пропусков нет.

Колонки: код претендента, рейтинг вуза окончания, средний балл диплома, опыт работы, возраст, рейтинг вуза по диплому, баллы за сертификаты, желаемая з/п и итоговая з/п от компании — последняя и будет ответом.

### 2. На основе файла создайте матрицу X с характеристиками (некоторые столбцы могут не понадобятся для характеристик, подумайте какие), и столбец y (из последнего столбца).

* отобразите первые строки матрицы X и столбца y на экране.

In [3]:
# короткие имена, чтобы удобнее было работать
data = data.rename(columns={
    'Код претендента': 'id',
    'Рейтинг университета окончания': 'uni_end',
    'Средний балл диплома': 'gpa',
    'Опыт работы (лет)': 'exp',
    'Возраст': 'age',
    'Рейтинг университета диплома': 'uni_diploma',
    'Баллы за сертификаты': 'cert',
    'Желаемая заработная плата': 'wanted_salary',
    'Итоговая зарплата от компании': 'salary',
})

# id — просто номер кандидата, на зарплату он не влияет, поэтому его не берём
features = ['uni_end', 'gpa', 'exp', 'age', 'uni_diploma', 'cert', 'wanted_salary']
X = data[features].values
y = data['salary'].values

print(X[:5])
print(y[:5])

[[8.20e+01 3.22e+00 1.00e+01 3.30e+01 8.40e+01 8.90e+01 2.37e+05]
 [4.00e+00 3.19e+00 3.00e+00 2.90e+01 1.60e+01 0.00e+00 1.55e+05]
 [7.00e+01 3.84e+00 6.00e+00 3.00e+01 7.00e+01 7.40e+01 2.12e+05]
 [9.80e+01 3.67e+00 2.00e+00 2.40e+01 9.70e+01 6.40e+01 1.40e+05]
 [9.40e+01 3.92e+00 2.00e+00 2.70e+01 9.90e+01 6.60e+01 1.46e+05]]
[237000 155000 197000 130000 136000]


Делим выборку на три части: 70 % train (учим), 15 % val (проверяем и правим модель), 15 % test (смотрим только в самом конце).

In [4]:
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.3, random_state=7)
X_vl, X_ts, y_vl, y_ts = train_test_split(X_rest, y_rest, test_size=0.5, random_state=7)
print('train', X_tr.shape, ' val', X_vl.shape, ' test', X_ts.shape)

train (560, 7)  val (120, 7)  test (120, 7)


### 3. Используя SAG, постройте на этих данных модель линейной регрессии

* выведите на экран полученное уравнение линиии

Признаки очень разного масштаба (gpa ≈ 4, желаемая з/п ≈ 250 000), поэтому перед SAG нормирую всё по train: $x' = (x - \text{mean}) / \text{std}$. После обучения перевожу коэффициенты обратно, чтобы уравнение было в нормальных единицах.

SAG: для каждого объекта помню его последний градиент, и шаг делаю по среднему из всех запомненных градиентов, а пересчитываю за шаг только один (для случайного объекта).

In [5]:
def sag(Xs, ys, lr=None, n_epochs=100, tol=1e-9, seed=1):
    n, m = Xs.shape
    Xb = np.hstack([np.ones((n, 1)), Xs])
    if lr is None:
        lr = 1 / (2 * (Xb ** 2).sum(axis=1).max())   # 1/L
    b = np.zeros(m + 1)
    memory = np.zeros((n, m + 1))   # запомненные градиенты
    avg = np.zeros(m + 1)           # их среднее
    rnd = np.random.RandomState(seed)
    for epoch in range(n_epochs):
        b_old = b.copy()
        for i in rnd.randint(0, n, size=n):   # n случайных объектов за эпоху
            new_g = 2 * (Xb[i] @ b - ys[i]) * Xb[i]
            avg += (new_g - memory[i]) / n
            memory[i] = new_g
            b = b - lr * avg
        if np.linalg.norm(b - b_old) < tol:   # веса перестали меняться
            break
    return b


def fit(X, y, **kw):
    mu, sd = X.mean(axis=0), X.std(axis=0)
    y_mu, y_sd = y.mean(), y.std()
    bs = sag((X - mu) / sd, (y - y_mu) / y_sd, **kw)
    b = bs[1:] * y_sd / sd                     # обратно в исходный масштаб
    b0 = y_mu + y_sd * bs[0] - (b * mu).sum()
    return np.concatenate([[b0], b])


def predict(b, X):
    return b[0] + X @ b[1:]


def show_line(b, names):
    parts = [f'{b[0]:.1f}'] + [f'({c:.3f})*{nm}' for c, nm in zip(b[1:], names)]
    print('y^ = ' + ' + '.join(parts))


b = fit(X_tr, y_tr)
show_line(b, features)

y^ = 40420.3 + (-163.692)*uni_end + (11101.546)*gpa + (9796.795)*exp + (75.790)*age + (-201.861)*uni_diploma + (229.382)*cert + (0.318)*wanted_salary


### 4. Проверьте веса на значимость, используя алгоритм и формулы из лекции

* код должен соотвествовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы
* какие веса у вас оказались незначимыми?

По лекции: $H_0: b_i = 0$, $H_1: b_i \ne 0$.

* $s_e^2 = \frac{1}{n-k-1}\sum(y_i - \hat y_i)^2$
* $C = s_e^2 (X^T X)^{-1}$ (в $X$ первый столбец из единиц)
* $S(b_i) = \sqrt{c_{ii}}$, $t_i = b_i / S(b_i)$
* если $t_i \in [t_{\alpha/2}, t_{1-\alpha/2}]$ → $H_0$ (признак не важен), иначе → $H_1$ (важен); $\alpha = 0.05$.

Проверяю на val. Убираю незначимые признаки **по одному** — каждый раз самый «слабый» (с минимальным $|t|$), потом переобучаю и проверяю снова. Так правильнее, чем выкидывать сразу все: после удаления одного признака значимость других может поменяться.

In [6]:
def t_test(b, X, y, alpha=0.05):
    n, k = X.shape
    Xb = np.hstack([np.ones((n, 1)), X])
    y_hat = Xb @ b
    s2 = ((y - y_hat) ** 2).sum() / (n - k - 1)
    C = s2 * np.linalg.inv(Xb.T @ Xb)
    t = b / np.sqrt(np.diag(C))
    left, right = st.t.ppf(alpha / 2, n - k - 1), st.t.ppf(1 - alpha / 2, n - k - 1)
    important = (t < left) | (t > right)
    return t, left, right, important


t, left, right, important = t_test(b, X_vl, y_vl)
print(f'интервал H0: [{left:.3f}, {right:.3f}]')
for nm, ti, imp in zip(['b0'] + features, t, important):
    print(f'{nm:>14}: t = {ti:8.3f}   ->  {"H1, важен" if imp else "H0, НЕ важен"}')

интервал H0: [-1.981, 1.981]
            b0: t =    2.532   ->  H1, важен
       uni_end: t =   -1.288   ->  H0, НЕ важен
           gpa: t =    6.987   ->  H1, важен
           exp: t =   11.104   ->  H1, важен
           age: t =    0.132   ->  H0, НЕ важен
   uni_diploma: t =   -1.701   ->  H0, НЕ важен
          cert: t =    8.393   ->  H1, важен
 wanted_salary: t =    9.255   ->  H1, важен


Две гипотезы нужны потому, что $b_i$ мы оценили по выборке, и он почти всегда получится не нулём даже у признака, который ни на что не влияет — из-за случайности в данных. Поэтому вопрос не «ноль или нет», а «можно ли объяснить такой $b_i$ случайностью». $H_0$ — можно (на самом деле $b_i = 0$), $H_1$ — нельзя (признак реально влияет). Одна из них обязательно верна, и $t$-статистика показывает, какая правдоподобнее: если $t$ далеко за пределами интервала, при $b_i = 0$ такое почти невозможно.

Незначимые на первом шаге: **age** ($t \approx 0.13$), **uni_end** ($t \approx -1.29$) и **uni_diploma** ($t \approx -1.70$). Удаляю самый слабый — age — и проверяю заново.

### 5. Если какие-то веса оказались незначимыми, перепишите матрицу характеристик X без соответствующих столбцов, обучите заново модель, используя SAG

In [7]:
cur = features.copy()

for step in range(1, len(features)):
    idx = [features.index(f) for f in cur]
    b_cur = fit(X_tr[:, idx], y_tr)
    t, left, right, important = t_test(b_cur, X_vl[:, idx], y_vl)
    print(f'шаг {step}: признаки {cur}')
    print('   t =', np.round(t[1:], 2))
    if important[1:].all():
        print('   все признаки значимы -> останавливаемся')
        break
    worst = cur[np.argmin(np.abs(t[1:]))]
    print(f'   удаляем {worst}\n')
    cur.remove(worst)

sel = [features.index(f) for f in cur]
X_tr2, X_vl2, X_ts2 = X_tr[:, sel], X_vl[:, sel], X_ts[:, sel]
b2 = fit(X_tr2, y_tr)
print()
show_line(b2, cur)

шаг 1: признаки ['uni_end', 'gpa', 'exp', 'age', 'uni_diploma', 'cert', 'wanted_salary']
   t = [-1.29  6.99 11.1   0.13 -1.7   8.39  9.25]
   удаляем age

шаг 2: признаки ['uni_end', 'gpa', 'exp', 'uni_diploma', 'cert', 'wanted_salary']
   t = [-1.29  7.02 17.74 -1.72  8.48  9.43]
   удаляем uni_end

шаг 3: признаки ['gpa', 'exp', 'uni_diploma', 'cert', 'wanted_salary']
   t = [  7.41  19.02 -10.62   8.73  10.5 ]
   все признаки значимы -> останавливаемся

y^ = 41079.5 + (11101.775)*gpa + (9825.892)*exp + (-355.858)*uni_diploma + (229.802)*cert + (0.322)*wanted_salary


Ушли **age** и **uni_end**, а uni_diploma после удаления uni_end стал значимым ($t$ с −1.7 до −10.6) — вот почему удалять нужно по одному. Оба дублируют другие признаки: возраст почти однозначно связан с опытом, а рейтинг вуза окончания почти совпадает с рейтингом вуза по диплому (проверка ниже). Модель не может разделить их вклад, поэтому они и получаются незначимыми. Свободный член и оставшиеся 5 признаков значимы.

In [8]:
print('corr(age, exp) =', round(np.corrcoef(data['age'], data['exp'])[0, 1], 3))
print('corr(uni_end, uni_diploma) =', round(np.corrcoef(data['uni_end'], data['uni_diploma'])[0, 1], 3))

corr(age, exp) = 0.952
corr(uni_end, uni_diploma) = 0.967


### 6. Проверьте модель на значимость по алгоритму из лекции

* код должен соответствовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы и что эти гипотезы собой подразумевают
* какая у вас получилась модель?
* подходит ли она для описания наших данных?

По лекции: $H_0: b_1 = \dots = b_k = 0$, $H_1: b_1^2 + \dots + b_k^2 > 0$.

$F = \dfrac{R^2/k}{(1-R^2)/(n-k-1)}$, где $R^2 = 1 - \dfrac{\sum(y_i - \hat y_i)^2}{\sum(y_i - \bar y)^2}$. Если $F \le F_{1-\alpha}$ → $H_0$ (модель незначима), если $F > F_{1-\alpha}$ → $H_1$ (значима). Проверяю на val.

In [9]:
def r_squared(y, y_hat):
    return 1 - ((y - y_hat) ** 2).sum() / ((y - y.mean()) ** 2).sum()

n, k = X_vl2.shape
R2 = r_squared(y_vl, predict(b2, X_vl2))
F = (R2 / k) / ((1 - R2) / (n - k - 1))
F_table = st.f.ppf(0.95, k, n - k - 1)
print(f'R2 = {R2:.4f}, F = {F:.1f}, F_0.95({k}, {n-k-1}) = {F_table:.3f}')
print('H1 -> модель значима' if F > F_table else 'H0 -> модель незначима')

R2 = 0.9844, F = 1440.2, F_0.95(5, 114) = 2.294
H1 -> модель значима


$H_0$ означает, что ни одна характеристика не влияет на ответ, т.е. модель ничем не лучше константы. $H_1$ — что хотя бы один вес не ноль (сумма квадратов весов больше нуля), т.е. модель что-то реально объясняет. Проверяем обе, потому что $R^2$ на выборке никогда не будет ровно 0 даже у пустой модели, и надо понять, не случайно ли он получился большим. F — отношение объяснённой части разброса к необъяснённой с поправкой на число признаков и наблюдений.

Получилась модель
$\hat y = b_0 + b_1\cdot gpa + b_2\cdot exp + b_3\cdot uni\_diploma + b_4\cdot cert + b_5\cdot wanted\_salary$
(коэффициенты — в выводе п. 5). $F$ больше табличного в сотни раз → $H_1$, модель значима и подходит для описания данных.

### 7. Рассчитайте парный коэффициент корреляции для каждой пары x_i, y, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* ответьте на вопрос "что в общем случае (а не в конкретно нашем) показывают парные коэффициенты корреляции?"
* выведите коэффициент корреляции на экран для каждой пары в виде: пара, значение коэффициента
* ответьте на вопрос "что полученные значения говорят о нашей модели?"
* какие характеристики оказывают на ответ наибольшее значение, какие наименьшее?

$r_{x,y} = \dfrac{\frac1n\sum(x_i - \bar x)(y_i - \bar y)}{\sigma_x \sigma_y}$, $\sigma = \sqrt{\frac1n \sum (\cdot - \text{mean})^2}$. Считаю на val для всех 7 исходных признаков.

In [10]:
def pair_corr(x, y):
    n = len(x)
    cov = ((x - x.mean()) * (y - y.mean())).sum() / n
    sx = np.sqrt(((x - x.mean()) ** 2).sum() / n)
    sy = np.sqrt(((y - y.mean()) ** 2).sum() / n)
    return cov / (sx * sy)

corrs = {f: pair_corr(X_vl[:, j], y_vl) for j, f in enumerate(features)}
for f, r in sorted(corrs.items(), key=lambda p: -abs(p[1])):
    print(f'({f}, salary)  ->  {r:+.3f}')

(wanted_salary, salary)  ->  +0.967
(exp, salary)  ->  +0.933
(age, salary)  ->  +0.855
(cert, salary)  ->  +0.298
(gpa, salary)  ->  +0.153
(uni_end, salary)  ->  -0.059
(uni_diploma, salary)  ->  -0.033


**В общем случае** парный коэффициент корреляции показывает, насколько сильно одна характеристика $x_i$ линейно связана с ответом $y$. Он от −1 до 1: чем больше по модулю — тем сильнее $x_i$ влияет на $y$, около нуля — влияния почти нет. Плюс — вместе растут, минус — один растёт, другой падает.

**Для нашей модели:** у wanted_salary и exp корреляция с зарплатой больше 0.9, т.е. зарплата почти линейно зависит от них — поэтому линейная модель и работает хорошо. У age тоже высокая корреляция, но это из‑за того, что возраст ≈ опыт; в модели он ничего не добавлял и был удалён. cert и gpa связаны с зарплатой слабо и положительно (≈ 0.30 и 0.15). Рейтинги вузов в паре с зарплатой почти не связаны (≈ −0.03…−0.06), но uni_diploma в модели всё равно значим: при одинаковых опыте и желаемой з/п выпускникам вузов с большим рейтингом компания платит немного меньше. Парная корреляция этого не видит, потому что смотрит только на одну пару.

**Больше всего влияют:** wanted_salary и exp. **Меньше всего:** uni_end и uni_diploma.

### 8. Вычислите коэффициент детерминации

* код должен соответствовать формуле из лекции
* выведите полученный коэффициент на экран
* что он говорит о нашей модели? насколько качественно она описала данные? лучше или хуже она, чем средняя линия?

In [11]:
print('R2 val  (итоговая, 5 признаков):', round(r_squared(y_vl, predict(b2, X_vl2)), 4))
print('R2 val  (все 7 признаков):      ', round(r_squared(y_vl, predict(b, X_vl)), 4))
print('R2 train (итоговая):            ', round(r_squared(y_tr, predict(b2, X_tr2)), 4))

R2 val  (итоговая, 5 признаков): 0.9844
R2 val  (все 7 признаков):       0.9841
R2 train (итоговая):             0.9858


$R^2 = 1 - \frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$ — сравнение ошибки нашей модели с ошибкой средней линии $\bar y$.

$R^2$ получился около 0.985, т.е. близко к 1: предсказания почти совпадают с реальными значениями, модель описывает данные очень хорошо и **намного лучше средней линии** (у средней линии $R^2 = 0$). После удаления age и uni_end качество не упало. На train и val $R^2$ одинаковый — модель не переобучилась.

### 9. Рассчитайте LOOCV кросс-валидации, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* объясните устно как он работает и зачем он нужен
* выведите LOOCV на экран
* ответьте на вопрос "что полученный LOOCV говорит о нашей модели? она идеально нам подходит, или в ней нужно что-то менять?"

**LOOCV** по лекции: по очереди каждое наблюдение становится test, на остальных $n-1$ учим модель (SAG), считаем ошибку $Error_i = y_i - \hat y_i$ на выкинутом наблюдении. В конце $LOOCV = \sqrt{\frac1n\sum Error_i^2}$.

Зачем: это способ понять, как модель будет работать на новых данных, не трогая тестовую выборку. Каждое наблюдение один раз оказывается «новым» для модели, а учится модель почти на всех данных, так что оценка получается честной. Делаю на train.

In [12]:
def LOOCV(X, y, n_epochs=40):
    errors = []
    for i in range(len(y)):
        X_train_i = np.delete(X, i, axis=0)
        y_train_i = np.delete(y, i)
        b_i = fit(X_train_i, y_train_i, n_epochs=n_epochs, seed=i)
        errors.append(y[i] - predict(b_i, X[i]))
    errors = np.array(errors)
    return np.sqrt((errors ** 2).mean())

loocv_value = LOOCV(X_tr2, y_tr)
print(f'LOOCV = {loocv_value:.0f} руб.')
print(f'средняя зарплата в train = {y_tr.mean():.0f}, это {loocv_value / y_tr.mean() * 100:.1f} % от неё')

LOOCV = 8938 руб.
средняя зарплата в train = 231195, это 3.9 % от неё


LOOCV — это средняя ошибка (в рублях), с которой модель предсказывает зарплату кандидата, которого не видела. Около 8.9 тыс. при средней зарплате ~231 тыс. — это ~4 %, ошибка относительно маленькая, значит на новых данных модель должна работать хорошо.

Но идеальной её назвать нельзя: если посмотреть на данные, итоговая зарплата часто либо равна желаемой, либо урезана на какой‑то процент — это не совсем линейная зависимость. Можно было бы попробовать добавить нелинейные признаки (например, произведение опыта и желаемой зарплаты) или другую модель. Мультиколлинеарность уже убрали, когда удалили age и uni_end.

### Тестирование на test

In [13]:
pred = predict(b2, X_ts2)
test_error = np.sqrt(((y_ts - pred) ** 2).mean())
print(f'ошибка на test (RMSE) = {test_error:.0f} руб.')
print(f'R2 на test = {r_squared(y_ts, pred):.4f}')

ошибка на test (RMSE) = 9043 руб.
R2 на test = 0.9846


Ошибка на test близка к LOOCV, т.е. кросс-валидация правильно предсказала, как модель поведёт себя на новых данных.